# Taller: App Streamlit para actualizar inventario + chatbot (Parte 3)

Objetivos:
- Construir una App (Streamlit) dentro de Databricks para actualizar en tiempo real la tabla `inventario_insumos_oficina`.
- Agregar un chatbot básico para consultas sobre la tabla.

Referencia: databricks-apps-cookbook
- `https://github.com/databricks-solutions/databricks-apps-cookbook/`

Nota: Esta app puede ejecutarse como Databricks App o desde un notebook (modo demo). Ajusta según tu entorno.



In [0]:
# Configuración rápida (usa el mismo catálogo/esquema/tabla de la Parte 1)

CATALOGO = f"databricks_workshop_rico" #cabia este a tu apellio
ESQUEMA = "gold"
TABLA = "inventario_insumos_oficina"

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE `{CATALOGO}`.`{ESQUEMA}`")
print(f"Usando {CATALOGO}.{ESQUEMA}.{TABLA}")


In [0]:
import streamlit as st
import pandas as pd
from databricks import sql
from databricks.sdk.core import Config
from databricks.sdk import WorkspaceClient ####


# -----------------------------
# CONFIGURACIÓN DEL CATÁLOGO/TABLA
# -----------------------------
CATALOGO = "latam_hunter"    # tu catálogo
ESQUEMA = "gold"             # tu esquema
TABLA = "inventario_insumos_oficina"  # tu tabla

TABLE_FULL_NAME = f"{CATALOGO}.{ESQUEMA}.{TABLA}"

# -----------------------------
# CONFIGURACIÓN DATBRICKS SQL
# -----------------------------
cfg = Config()  # Debe tener DATABRICKS_HOST y DATABRICKS_TOKEN configurados en el entorno

st.set_page_config(page_title="Inventario Oficina", layout="wide")
st.title("Inventario de Insumos de Oficina")

# --- Conexión SQL ---
@st.cache_resource(ttl="1h")  # Cachea la conexión
def get_connection(http_path: str):
    return sql.connect(
        server_hostname=cfg.host,
        http_path=http_path,
        credentials_provider=lambda: cfg.authenticate,
    )

# --- Leer tabla ---
def read_table(table_name: str, conn):
    with conn.cursor() as cursor:
        query = f"SELECT * FROM {table_name}"
        cursor.execute(query)
        return cursor.fetchall_arrow().to_pandas()

# --- Actualizar stock ---
def update_stock(table_name: str, item_id: str, nuevo_stock: int, conn):
    with conn.cursor() as cursor:
        query = f"""
        UPDATE {table_name}
        SET stock_actual = {nuevo_stock}
        WHERE item_id = '{item_id}'
        """
        cursor.execute(query)

# -----------------------------
# INPUT HTTP PATH
# -----------------------------
http_path_input = st.text_input(
    "Enter your Databricks HTTP Path:", placeholder="/sql/1.0/warehouses/xxxxxx"
)

if http_path_input:
    # Crear conexión
    conn = get_connection(http_path_input)
    df = read_table(TABLE_FULL_NAME, conn)
    
    st.subheader("Vista previa")
    st.dataframe(df.head(50))
    
    # --- Actualizar stock ---
    st.subheader("Actualizar stock")
    item_id = st.text_input("Item ID (ej.: ITM0001)", key="update_id")
    nuevo_stock = st.number_input("Nuevo stock", min_value=0, step=1, key="update_stock")
    
    if st.button("Actualizar"):
        if item_id:
            update_stock(TABLE_FULL_NAME, item_id, nuevo_stock, conn)
            st.success(f"Stock actualizado para {item_id} → {nuevo_stock}")
            df = read_table(TABLE_FULL_NAME, conn)
            st.dataframe(df.head(50))
        else:
            st.warning("Ingresa un Item ID válido.")
    
    # --- Filtros ---
    st.subheader("Filtrar por categoría / subcategoría")
    categorias = ["(todas)"] + df["categoria"].dropna().unique().tolist()
    cat_sel = st.selectbox("Categoría", options=categorias, key="cat")
    
    sub_list = []
    if cat_sel != "(todas)":
        sub_list = ["(todas)"] + df[df["categoria"] == cat_sel]["subcategoria"].dropna().unique().tolist()
    else:
        sub_list = ["(todas)"]
    sub_sel = st.selectbox("Subcategoría", options=sub_list, key="subcat")
    
    # Filtrar DataFrame
    df_filtered = df.copy()
    if cat_sel != "(todas)":
        df_filtered = df_filtered[df_filtered["categoria"] == cat_sel]
    if sub_sel != "(todas)":
        df_filtered = df_filtered[df_filtered["subcategoria"] == sub_sel]
    
    st.write("Resultados filtrados:")
    st.dataframe(df_filtered.head(200))

    # --- Gráfica de los top productos filtrados ---
    st.subheader("Top productos filtrados por stock")

    if not df_filtered.empty:
        # Ordenar por stock_actual descendente
        top_products = df_filtered.sort_values(by="stock_actual", ascending=False).head(20)
        
        # Crear gráfico de barras: nombre del producto vs stock_actual
        st.bar_chart(
            top_products.set_index("nombre")["stock_actual"],
            use_container_width=True
        )
    else:
        st.info("No hay productos para mostrar con los filtros actuales.")


    # --- Gráfica ejemplo ---
    st.header("Hello world!!!")
    apps = st.slider("Number of apps", max_value=60, value=10)
    chart_data = pd.DataFrame({'y':[2 ** x for x in range(apps)]})
    st.bar_chart(chart_data, height=500, width=min(100+50*apps, 1000), 
                 use_container_width=False, x_label="Apps", y_label="Fun with data")


# -----------------------------
# Instrucciones para agregar el chatbot Genie a la app
# -----------------------------

### 🤖 Instrucciones para agregar el chatbot Genie

1. **Configura tu Genie Space en Databricks:**
   - Ve a la sección de Genie en tu workspace de Databricks.
   - Crea un nuevo espacio Genie o usa uno existente.
   - Copia el `Genie Space ID` que se mostrará en la configuración.

2. **Agrega tu Genie Space ID en el código:**
   - Busca la variable `genie_space_id` en el código.
   - Reemplaza `"Genie_ID"` por el ID real de tu espacio Genie.

3. **Asegúrate de tener permisos y el token configurado:**
   - Debes tener configuradas las variables de entorno `DATABRICKS_HOST` y `DATABRICKS_TOKEN` para la autenticación.

4. **Utiliza el chat en la app:**
   - Escribe tu pregunta en el cuadro de chat en la parte inferior de la app.
   - El asistente responderá usando Genie, mostrando resultados y código SQL generado si aplica.

5. **Personaliza las instrucciones de Genie (opcional):**
   - En la configuración de tu espacio Genie, puedes agregar instrucciones específicas en español para mejorar las respuestas del asistente.

> **Nota:** Si tienes problemas con la conexión o el ID, revisa la configuración y permisos de tu workspace.

---

In [0]:

# -----------------------------         ######
# Configuración Workspace
# -----------------------------
w = WorkspaceClient()
genie_space_id = "Genie_ID"  configura tu genie ID ######


# -----------------------------
# Indicador para el chat
# -----------------------------
st.markdown("---")  # Línea separadora opcional
st.subheader("💬 Habla con tus datos / Pregunta insights")

# -----------------------------
# Chat con Genie
# -----------------------------

def display_message(message):
    if "content" in message:
        st.markdown(message["content"])
    if "data" in message:
        st.dataframe(message["data"])
    if "code" in message:
        with st.expander("Show generated code"):
            st.code(message["code"], language="sql", wrap_lines=True)


def get_query_result(statement_id):
    # For simplicity, let's say data fits in one chunk, query.manifest.total_chunk_count = 1

    result = w.statement_execution.get_statement(statement_id)
    return pd.DataFrame(
        result.result.data_array, columns=[i.name for i in result.manifest.schema.columns]
    )


def process_genie_response(response):
    for i in response.attachments:
        if i.text:
            message = {"role": "assistant", "content": i.text.content}
            display_message(message)
        elif i.query:
            data = get_query_result(response.query_result.statement_id)
            message = {
                "role": "assistant", "content": i.query.description, "data": data, "code": i.query.query
            }
            display_message(message)


if prompt := st.chat_input("Ask your question..."):
    # Refer to actual app code for chat history persistence on rerun

    st.chat_message("user").markdown(prompt)

    with st.chat_message("assistant"):
        if st.session_state.get("conversation_id"):
            conversation = w.genie.create_message_and_wait(
                genie_space_id, st.session_state.conversation_id, prompt
            )
            process_genie_response(conversation)
        else:
            conversation = w.genie.start_conversation_and_wait(genie_space_id, prompt)
            process_genie_response(conversation)

